# La Plata: elecciones legislativas, 2013-2025

Mismo distrito que los notebooks anteriores (La Plata, Provincia de Buenos Aires, Generales), ahora para las elecciones legislativas: **2013, 2017, 2021 y 2025**. Tres niveles:

- **nacional**: Diputados Nacionales + Senadores Nacionales
- **provincial**: Diputados Provinciales + Senadores Provinciales
- **municipal**: Concejales

El `idCargo` de cada cargo se resolvió igual que en el notebook anterior: probando valores contra `resultado/totalizadocsv` y leyendo `cargo_nombre`. Para La Plata, estable en los cuatro años:

| idCargo | cargo | nivel |
|---|---|---|
| 2 | Senador Nacional | nacional |
| 3 | Diputado(s) Nacional(es) | nacional |
| 5 | Senadores Provinciales | provincial |
| 6 | Diputado(s) Provincial(es) | provincial |
| 10 | Concejales | municipal |

**Limitación real encontrada en 2025**: en la Provincia de Buenos Aires las elecciones legislativas provinciales y municipales de 2025 se desdoblaron de la nacional (se votaron en fecha distinta, administradas por la Junta Electoral provincial, no por la nacional). Este sistema (del Ministerio del Interior, nacional) **solo tiene Diputados Nacionales para La Plata/2025** — ni Senadores, ni Diputados Provinciales, ni Concejales están disponibles acá para ese año. Se confirmó barriendo `idCargo` 1-25 sin encontrar nada más.

In [ ]:
import io
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from electoral.client import ResultadosClient, ResultadosNoDisponibles
from electoral.models import ResultadoElectoral

REPO = Path.cwd().parent
client = ResultadosClient(cache_dir=REPO / "data")

ANIOS = [2013, 2017, 2021, 2025]

# (nivel, categoria_id, nombre) — nivel decide la carpeta de caché (data/<anio>/<nivel>/);
# nombre es solo para identificar el cargo en los prints de este notebook.
CARGOS = [
    ("nacional", 2, "senador_nacional"),
    ("nacional", 3, "diputados_nacionales"),
    ("provincial", 5, "senadores_provinciales"),
    ("provincial", 6, "diputados_provinciales"),
    ("municipal", 10, "concejales"),
]

LA_PLATA = dict(
    tipo_eleccion=2,  # Generales
    distrito_id=2,  # Buenos Aires
    seccion_provincial_id=8,  # Sección Capital
    seccion_id=63,  # La Plata
)

## 1. Traer el CSV oficial de cada (año, cargo) disponible

In [ ]:
dataframes = {}
faltantes = []
for anio in ANIOS:
    for nivel, categoria_id, nombre in CARGOS:
        try:
            csv_bytes = client.get_resultados_csv(
                anio_eleccion=anio, categoria_nombre=nivel, categoria_id=categoria_id, **LA_PLATA
            )
        except ResultadosNoDisponibles:
            faltantes.append((anio, nombre))
            continue
        df = pd.read_csv(io.BytesIO(csv_bytes), low_memory=False)
        df["agrupacion_nombre"] = df["agrupacion_nombre"].str.strip()
        dataframes[(anio, categoria_id)] = {"nivel": nivel, "nombre": nombre, "df": df}
        print(f"{anio}/{nivel}/{nombre} (idCargo={categoria_id}): {len(df)} filas, "
              f"{df['mesa_id'].nunique()} mesas, cargo_nombre={df['cargo_nombre'].unique()}")

print("\nno disponibles (esperado, no todo se vota todos los años):")
for anio, nombre in faltantes:
    print(f"  {anio}/{nombre}")

## 2. Resultado de cada (año, cargo) disponible

In [ ]:
for (anio, categoria_id), info in dataframes.items():
    df = info["df"]
    positivos = (
        df[df["votos_tipo"] == "POSITIVO"]
        .groupby("agrupacion_nombre")["votos_cantidad"]
        .sum()
        .sort_values(ascending=False)
    )
    print(f"=== {anio}/{info['nivel']}/{info['nombre']} ===")
    print(positivos.head(5).to_string())
    print()

## 3. Validar contra el agregado de la API (JSON)


In [ ]:
def normalizar_id(x):
    x = str(x)
    return (x.lstrip("0") or "0") if x.isdigit() else x


for (anio, categoria_id), info in dataframes.items():
    df = info["df"]
    positivos_csv = df[df["votos_tipo"] == "POSITIVO"].groupby("agrupacion_id")["votos_cantidad"].sum()
    positivos_csv.index = positivos_csv.index.map(normalizar_id)

    raw = client.get_resultados(
        anio_eleccion=anio, categoria_nombre=info["nivel"], categoria_id=categoria_id, **LA_PLATA
    )
    resultado = ResultadoElectoral.from_json(raw)
    positivos_api = {
        normalizar_id(a.id_agrupacion): a.votos for a in resultado.valores_totalizados_positivos
    }

    agrupaciones = sorted(set(positivos_csv.index) | set(positivos_api))
    diffs = [ag for ag in agrupaciones if positivos_csv.get(ag, 0) != positivos_api.get(ag, 0)]
    estado = "OK" if not diffs else f"AGREGADO JSON NO CONFIABLE (difiere en {len(diffs)} agrupaciones)"
    print(f"{anio}/{info['nivel']}/{info['nombre']}: {estado}")

## 4. Tabla de agrupaciones legislativas


In [ ]:
filas_agrupaciones = []
for (anio, categoria_id), info in dataframes.items():
    raw = client.get_resultados(
        anio_eleccion=anio, categoria_nombre=info["nivel"], categoria_id=categoria_id, **LA_PLATA
    )
    resultado = ResultadoElectoral.from_json(raw)
    for a in resultado.valores_totalizados_positivos:
        filas_agrupaciones.append(
            {"anio": anio, "agrupacion": a.nombre_agrupacion, "nivel": info["nivel"]}
        )

df_agrupaciones_legislativas = (
    pd.DataFrame(filas_agrupaciones)
    .drop_duplicates()
    .sort_values(["anio", "nivel", "agrupacion"])
    .reset_index(drop=True)
)

destino = REPO / "data" / "agrupaciones"
destino.mkdir(parents=True, exist_ok=True)
df_agrupaciones_legislativas.to_csv(destino / "agrupaciones_legislativas.csv", index=False)

print(f"{len(df_agrupaciones_legislativas)} filas -> {destino / 'agrupaciones_legislativas.csv'}")
df_agrupaciones_legislativas.head(10)

## 5. Estado final del caché en disco


In [ ]:
for anio in ANIOS:
    for nivel in {n for n, _, _ in CARGOS}:
        carpeta = REPO / "data" / str(anio) / nivel
        archivos = sorted(p.name for p in carpeta.iterdir()) if carpeta.exists() else []
        print(f"{anio}/{nivel}: {len(archivos)} archivos")